# VITALIS: Camera-Based Heart Rate Estimation, Validated Against ECG

This notebook estimates heart rate from facial video using **remote photoplethysmography (rPPG)** with the **POS** algorithm, and validates the result against simultaneous **ECG** recordings on a 27-subject dataset.

**Pipeline**

1. Read each skin-region video and compute the mean R, G, B value per frame.
2. Extract the pulse signal with POS, then band-pass filter it (0.7-3.0 Hz).
3. Detect R-peaks in the ECG to build a reference heart rate.
4. Compare camera and ECG heart rate in non-overlapping 30-second windows (MAE, RMSE, correlation, Bland-Altman).

> **Disclaimer:** this is a research prototype for education and experimentation. It is **not** a medical device and must not be used for diagnosis or treatment decisions.

The signal-processing code lives in `vitalis_core.py` so the notebook and the live webcam script (`vitalis_live.py`) share exactly the same methods.

## 1. Setup and configuration

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import vitalis_core as vc

# Paths are relative so the notebook runs on any machine.
# Override with environment variables if your data lives elsewhere:
#   VITALIS_DATASET=/path/to/Dataset_rPPG-10
DATASET_PATH = Path(os.environ.get("VITALIS_DATASET", "dataset/Dataset_rPPG-10"))
RESULTS_PATH = Path(os.environ.get("VITALIS_RESULTS", "results"))
CACHE_PATH = RESULTS_PATH / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)

ECG_FS = 1000           # Hz (inferred, see section 3)
WINDOW_SECONDS = 30     # evaluation window length
HR_LOW, HR_HIGH = 0.7, 3.0   # Hz (42-180 BPM)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH.resolve()}. "
        "See the README for download instructions."
    )

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_PATH)

## 2. Dataset inventory

In [ ]:
def subject_number(name):
    return int(name.split("_")[1])

subjects = sorted(
    [p.name for p in DATASET_PATH.iterdir() if p.is_dir()],
    key=subject_number,
)

inventory = pd.DataFrame([
    {
        "subject": s,
        "videos": len(list((DATASET_PATH / s).glob("*.avi"))),
        "ecg_files": len(list((DATASET_PATH / s).glob("*.npy"))),
    }
    for s in subjects
])

display(inventory)
print("Subjects:", len(subjects))
print("Total videos:", inventory["videos"].sum())
print("Total ECG files:", inventory["ecg_files"].sum())

## 3. Subject metadata and ECG sampling rate

The dataset ships with an Excel sheet of subject characteristics (age, sex, skin tone, health condition, ...). Two subjects are labelled with an arrhythmia; they are reported separately because irregular rhythm makes average heart rate a less meaningful target.

**ECG sampling rate:** no documentation file was found in the dataset folder. Each ECG has 600,000 samples and each video lasts 600 s, which implies **1000 Hz**. This is an inference from the recording length, not a documented value. Please verify it against the dataset paper.

In [ ]:
excel_files = sorted(DATASET_PATH.glob("*.xlsx"))
if not excel_files:
    raise FileNotFoundError("No metadata Excel file found in the dataset folder.")

raw_meta = pd.read_excel(excel_files[0])
meta = raw_meta.set_index("Characteristics").T
meta.index = [i.replace(" ", "_") for i in meta.index]   # "Subject 1" -> "Subject_1"
meta.index.name = "subject"
meta["arrhythmia"] = meta["Health Condition"].astype(str).str.strip().str.upper() != "N"

print("Subjects with metadata:", len(meta))
print("Subjects WITHOUT metadata:", [s for s in subjects if s not in meta.index])
print("Subjects flagged with arrhythmia:", meta.index[meta["arrhythmia"]].tolist())

# Sanity check of the inferred ECG sampling rate
lengths = {}
for s in subjects:
    f = sorted((DATASET_PATH / s).glob("*_ECG.npy"))
    if f:
        lengths[s] = len(np.load(f[0], allow_pickle=False))
print("\nECG lengths (unique):", sorted(set(lengths.values())),
      "-> duration at 1000 Hz:", sorted({n / ECG_FS for n in lengths.values()}), "s")

display(meta.head())

## 4. Single-subject walkthrough (Subject 1, forehead video)

Before running every subject, here is the full pipeline on one recording so each step can be inspected.

In [ ]:
def load_rgb_cached(video_path):
    """Read a video once and cache its RGB trace (decoding is the slow part)."""
    cache_file = CACHE_PATH / f"{video_path.stem}.npz"
    if cache_file.exists():
        data = np.load(cache_file)
        return float(data["fps"]), data["rgb"]
    fps, rgb = vc.extract_rgb_signal(video_path)
    np.savez_compressed(cache_file, fps=fps, rgb=rgb)
    return fps, rgb


demo_subject = "Subject_1"
demo_dir = DATASET_PATH / demo_subject
demo_video = sorted(demo_dir.glob("*Forehead*.avi"))[0]
demo_ecg_file = sorted(demo_dir.glob("*_ECG.npy"))[0]

fps, rgb = load_rgb_cached(demo_video)
pulse = vc.bandpass(vc.pos_algorithm(rgb, fps), fps, HR_LOW, HR_HIGH)

ecg = np.load(demo_ecg_file, allow_pickle=False)
ecg_filt = vc.filter_ecg(ecg, ECG_FS)
peaks = vc.detect_r_peaks(ecg_filt, ECG_FS)

# first 30-second window
win = int(WINDOW_SECONDS * fps)
hr_cam = vc.estimate_hr(pulse[:win], fps, HR_LOW, HR_HIGH)
hr_ref = vc.hr_from_peaks(peaks, ECG_FS, 0, WINDOW_SECONDS)
print(f"{demo_video.name}: camera {hr_cam:.1f} BPM | ECG {hr_ref:.1f} BPM "
      f"| error {hr_cam - hr_ref:+.1f} BPM")

fig, axes = plt.subplots(3, 1, figsize=(13, 9))

t = np.arange(win) / fps
axes[0].plot(t, pulse[:win])
axes[0].set(title="POS rPPG signal (first 30 s)", xlabel="Time (s)", ylabel="Amplitude (a.u.)")

n = 10 * ECG_FS
p10 = peaks[peaks < n]
axes[1].plot(np.arange(n) / ECG_FS, ecg_filt[:n], label="Filtered ECG")
axes[1].scatter(p10 / ECG_FS, ecg_filt[p10], color="red", zorder=3, label="R-peaks")
axes[1].set(title="ECG with detected R-peaks (first 10 s)", xlabel="Time (s)", ylabel="Amplitude")
axes[1].legend()

freqs, power = vc._spectrum(pulse[:win], fps)
band = (freqs >= HR_LOW) & (freqs <= HR_HIGH)
axes[2].plot(freqs[band] * 60, power[band])
axes[2].axvline(hr_ref, color="green", ls="--", label=f"ECG {hr_ref:.1f} BPM")
axes[2].axvline(hr_cam, color="red", ls=":", label=f"Camera {hr_cam:.1f} BPM")
axes[2].set(title="rPPG power spectrum", xlabel="Heart rate (BPM)", ylabel="Power")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_PATH / "single_subject_walkthrough.png", dpi=150)
plt.show()

## 5. Full evaluation: camera vs ECG, all subjects

For every video, the pulse signal and the ECG are split into **non-overlapping 30-second windows**. In each window the camera heart rate (spectral peak) is compared with the ECG heart rate (mean RR interval over the same seconds). This gives many paired measurements per subject instead of a single average.

Videos that cannot be opened are recorded in a failure list instead of stopping the run.

In [ ]:
rows, failures = [], []

for subject in subjects:
    sdir = DATASET_PATH / subject
    ecg_files = sorted(sdir.glob("*_ECG.npy"))
    if not ecg_files:
        failures.append((subject, "no ECG file"))
        continue

    ecg = np.load(ecg_files[0], allow_pickle=False)
    ecg_peaks = vc.detect_r_peaks(vc.filter_ecg(ecg, ECG_FS), ECG_FS)

    for video in sorted(sdir.glob("*.avi")):
        match = re.search(r"(Cheek1|Cheek2|Forehead)", video.name)
        region = match.group(1) if match else "Unknown"

        try:
            fps, rgb = load_rgb_cached(video)
            pulse = vc.bandpass(vc.pos_algorithm(rgb, fps), fps, HR_LOW, HR_HIGH)
        except Exception as e:
            failures.append((video.name, str(e)))
            continue

        win = int(WINDOW_SECONDS * fps)
        for start in range(0, len(pulse) - win + 1, win):
            seg = pulse[start:start + win]
            t0 = start / fps
            rows.append({
                "subject": subject,
                "region": region,
                "window_start_s": t0,
                "hr_rppg": vc.estimate_hr(seg, fps, HR_LOW, HR_HIGH),
                "hr_ecg": vc.hr_from_peaks(ecg_peaks, ECG_FS, t0, t0 + WINDOW_SECONDS),
                "snr_db": vc.pulse_snr(seg, fps, HR_LOW, HR_HIGH),
            })
    print("done:", subject)

results = pd.DataFrame(rows).dropna(subset=["hr_rppg", "hr_ecg"]).reset_index(drop=True)
results["error"] = results["hr_rppg"] - results["hr_ecg"]
results["abs_error"] = results["error"].abs()
results["arrhythmia"] = results["subject"].map(meta["arrhythmia"]).fillna(False).astype(bool)

results.to_csv(RESULTS_PATH / "window_results.csv", index=False)

print(f"\nValid windows: {len(results)} from {results['subject'].nunique()} subjects")
print("Failed items:", len(failures))
for name, reason in failures:
    print("  -", name, "->", reason[:90])

## 6. Accuracy metrics

In [ ]:
def summarize(df):
    e = df["error"]
    r = stats.pearsonr(df["hr_rppg"], df["hr_ecg"])[0] if len(df) > 2 else np.nan
    return pd.Series({
        "windows": len(df),
        "MAE (BPM)": e.abs().mean(),
        "RMSE (BPM)": np.sqrt((e ** 2).mean()),
        "Bias (BPM)": e.mean(),
        "Pearson r": r,
        "Within 5 BPM (%)": (e.abs() <= 5).mean() * 100,
    })

normal = results[~results["arrhythmia"]]

overall = pd.DataFrame({
    "All subjects": summarize(results),
    "Excluding arrhythmia": summarize(normal),
    "Arrhythmia only": summarize(results[results["arrhythmia"]]),
}).T
print("Overall")
display(overall.round(2))

by_region = normal.groupby("region").apply(summarize, include_groups=False)
print("By skin region (excluding arrhythmia)")
display(by_region.round(2))

overall.round(3).to_csv(RESULTS_PATH / "metrics_overall.csv")
by_region.round(3).to_csv(RESULTS_PATH / "metrics_by_region.csv")

## 7. Agreement plots

In [ ]:
colors = {"Cheek1": "tab:blue", "Cheek2": "tab:orange", "Forehead": "tab:green"}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: camera vs ECG
for region, g in normal.groupby("region"):
    axes[0].scatter(g["hr_ecg"], g["hr_rppg"], s=14, alpha=0.5,
                    color=colors.get(region), label=region)
lims = [40, 130]
axes[0].plot(lims, lims, "k--", lw=1, label="Perfect agreement")
axes[0].set(xlim=lims, ylim=lims, xlabel="ECG heart rate (BPM)",
            ylabel="Camera (rPPG) heart rate (BPM)",
            title="Camera vs ECG (30 s windows)")
axes[0].legend()

# Bland-Altman
mean_hr = (normal["hr_rppg"] + normal["hr_ecg"]) / 2
bias = normal["error"].mean()
sd = normal["error"].std()
axes[1].scatter(mean_hr, normal["error"], s=14, alpha=0.4)
axes[1].axhline(bias, color="red", label=f"Bias {bias:+.2f}")
axes[1].axhline(bias + 1.96 * sd, color="gray", ls="--",
                label=f"+1.96 SD ({bias + 1.96 * sd:+.1f})")
axes[1].axhline(bias - 1.96 * sd, color="gray", ls="--",
                label=f"-1.96 SD ({bias - 1.96 * sd:+.1f})")
axes[1].set(xlabel="Mean of camera and ECG (BPM)",
            ylabel="Camera - ECG (BPM)", title="Bland-Altman plot")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_PATH / "agreement_plots.png", dpi=150)
plt.show()

## 8. Where does it fail?

Per-subject error and a breakdown by skin tone and skin obstruction show whether failures are random or systematic. Group sizes are small (a few subjects each), so treat these as hints, not conclusions.

In [ ]:
per_subject = (
    results.groupby("subject")["abs_error"].mean()
    .sort_values(ascending=False)
)
print("Highest-error subjects (MAE, BPM):")
display(per_subject.head(8).round(2).to_frame("MAE"))

fig, ax = plt.subplots(figsize=(13, 4))
order = sorted(per_subject.index, key=subject_number)
ax.bar([s.replace("Subject_", "S") for s in order], per_subject[order])
ax.set(ylabel="MAE (BPM)", title="Mean absolute error per subject")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_PATH / "error_per_subject.png", dpi=150)
plt.show()

merged = normal.join(meta[["Skin Tone (FitzPatrick)", "Skin obstruction"]], on="subject")
for col in ["Skin Tone (FitzPatrick)", "Skin obstruction"]:
    table = merged.groupby(col).agg(
        subjects=("subject", "nunique"),
        windows=("error", "size"),
        MAE=("abs_error", "mean"),
    ).round(2)
    print(f"\nMAE by {col}")
    display(table)

# Does signal quality predict reliability?
corr = normal[["snr_db", "abs_error"]].corr().iloc[0, 1]
print(f"\nCorrelation between signal quality (SNR) and absolute error: {corr:.2f}")

## 9. Results summary and limitations

*Fill in after running the notebook on the full dataset:*

- Overall MAE / RMSE / correlation: see the tables in section 6.
- Best and worst skin region: see the by-region table.
- Failure cases: see section 8.

**Limitations**

- Small dataset (27 subjects, one recording session each), resting condition, controlled lighting.
- Videos are 64x64 pre-cropped skin regions; real webcam use needs face detection and tracking, which adds motion artefacts.
- ECG sampling rate (1000 Hz) is inferred from recording length.
- ECG reference uses simple R-peak detection; it is not manually verified.
- Subject 4 videos could not be opened in the original run (see the failure list above), so that subject is not evaluated.
- Spectral-peak estimation can lock onto a harmonic or motion noise; low-SNR windows should be treated as unreliable.

**Not a medical device.** Do not use these results for any clinical decision.